# 1. 2025 scraping

This notebook wraps the resumable `commentgap-scrape` command-line collector for a full publication year. It can discover stories, resume a crawl, validate the output, export legacy tables, and summarize the collected Parquet data.

The collection cells are disabled by default. Set the corresponding flags in the configuration cell before running them. The crawl requires `COMMENTGAP_USER_AGENT`, `COMMENTGAP_CONTACT`, and `COMMENTGAP_HASH_KEY` in the environment; the hash key is never stored by this notebook. Run the notebook from the repository root.

In [ ]:
from pathlib import Path
import json
import os
import sys

import duckdb
import pandas as pd
from IPython.display import display, Markdown
from commentgap_analysis.category_labels import translate_news_category
from commentgap_scraper.notebook_helpers import (
    parquet_files,
    parquet_relation,
    require_collection_credentials,
    run_scraper,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the commentgap repository root.")

YEAR = 2025
OUTPUT_DIR = PROJECT_ROOT / "data" / f"scrape_{YEAR}"
USER_AGENT = os.environ.get("COMMENTGAP_USER_AGENT", "CommentGap academic research crawler")
CONTACT = os.environ.get("COMMENTGAP_CONTACT", "")

# Collection actions. Leave these False to inspect an existing collection.
RUN_DISCOVER = False
RUN_CRAWL = False
RUN_VALIDATE = False
RUN_EXPORT_LEGACY = False

# Crawl options. CRAWL_LIMIT=None means the complete discovered year.
CRAWL_LIMIT = None
CRAWL_SELECTION = "chronological"
SELECTION_SEED = YEAR
RETRY_FAILED = False
VALIDATE_ALLOW_INCOMPLETE = False

print(f"Year: {YEAR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Python: {sys.executable}")

## Run the collector

The commands below call the same Python module exposed by the `commentgap-scrape` script. Crawling is resumable, so rerunning the crawl cell continues pending or interrupted work. A non-zero crawl status is allowed through so that partial output can still be validated and summarized.

In [ ]:
if RUN_DISCOVER:
    require_collection_credentials(CONTACT)
    run_scraper("discover", "--user-agent", USER_AGENT, "--contact", CONTACT, project_root=PROJECT_ROOT, year=YEAR, output_dir=OUTPUT_DIR)

if RUN_CRAWL:
    require_collection_credentials(CONTACT, include_hash_key=True)
    crawl_arguments = ["crawl", "--user-agent", USER_AGENT, "--contact", CONTACT]
    if CRAWL_LIMIT is not None:
        crawl_arguments += ["--limit", CRAWL_LIMIT, "--selection", CRAWL_SELECTION, "--selection-seed", SELECTION_SEED]
    if RETRY_FAILED:
        crawl_arguments.append("--retry-failed")
    run_scraper(*crawl_arguments, allow_failure=True, project_root=PROJECT_ROOT, year=YEAR, output_dir=OUTPUT_DIR)

if RUN_VALIDATE:
    validation_arguments = ["validate"]
    if VALIDATE_ALLOW_INCOMPLETE:
        validation_arguments.append("--allow-incomplete")
    run_scraper(*validation_arguments, project_root=PROJECT_ROOT, year=YEAR, output_dir=OUTPUT_DIR)

if RUN_EXPORT_LEGACY:
    run_scraper("export-legacy", project_root=PROJECT_ROOT, year=YEAR, output_dir=OUTPUT_DIR)

## Load the collected tables

The scraper stores article, forum, and comment records as month-partitioned Parquet files. The helper below discovers the files that are present, so it also works for a partial or month-scoped collection.

In [ ]:
TABLES = {name: parquet_relation(OUTPUT_DIR, name) for name in ("articles", "forums", "comments", "forum_pages")}
display(pd.DataFrame({"table": list(TABLES), "files": [len(parquet_files(OUTPUT_DIR, name)) for name in TABLES], "available": [TABLES[name] is not None for name in TABLES]}))

if TABLES["articles"] is None:
    raise FileNotFoundError(f"No article Parquet files found under {OUTPUT_DIR}. Run discover/crawl first or update OUTPUT_DIR.")

articles = TABLES["articles"]
forums = TABLES["forums"]
comments = TABLES["comments"]
empty_forums = "(SELECT CAST(NULL AS VARCHAR) AS story_id, CAST(0 AS BIGINT) AS observed_unique_count WHERE FALSE)"
empty_comment_counts = "(SELECT CAST(NULL AS VARCHAR) AS story_id, CAST(0 AS BIGINT) AS comment_rows, CAST(0 AS BIGINT) AS published_comments, CAST(0 AS BIGINT) AS deleted_comments WHERE FALSE)"
comment_counts = (
    f"(SELECT story_id, COUNT(*) AS comment_rows, \
COUNT(*) FILTER (WHERE lifecycle_status = 'Published') AS published_comments, \
COUNT(*) FILTER (WHERE lifecycle_status = 'Deleted') AS deleted_comments \
FROM {comments} GROUP BY story_id)"
    if comments
    else empty_comment_counts
)
print("Loaded relations:", ", ".join(name for name, relation in TABLES.items() if relation))

## Collection overview

In [ ]:
overview = duckdb.sql(f"""
        SELECT
            COUNT(*) AS stories,
            COUNT(*) FILTER (WHERE NULLIF(TRIM(body), '') IS NOT NULL) AS stories_with_body,
            COUNT(DISTINCT NULLIF(TRIM(section_1), '')) AS primary_sections,
            MIN(try_cast(NULLIF(published_at, '') AS TIMESTAMP)) AS first_published_at,
            MAX(try_cast(NULLIF(published_at, '') AS TIMESTAMP)) AS last_published_at
        FROM {articles}
    """).df()
display(overview)

## Stories, forums, and comments by month

In [ ]:
monthly = duckdb.sql(f"""
    SELECT
        a.year,
        a.month,
        COUNT(*) AS stories,
        COUNT(f.story_id) AS forum_records,
        COALESCE(SUM(f.observed_unique_count), 0) AS observed_comments,
        COALESCE(SUM(c.comment_rows), 0) AS stored_comments,
        COALESCE(SUM(c.published_comments), 0) AS published_comments,
        COALESCE(SUM(c.deleted_comments), 0) AS deleted_comments
    FROM {articles} AS a
    LEFT JOIN {forums or empty_forums} AS f USING (story_id)
    LEFT JOIN {comment_counts} AS c USING (story_id)
    GROUP BY a.year, a.month
    ORDER BY a.year, a.month
""").df()
display(monthly)

## Stories by primary section

`section_1` is the first section recorded by the scraper. A story can also have `section_2` and `section_3`; this table uses the primary section to keep each story in one row.

In [ ]:
by_section = duckdb.sql(f"""
    SELECT
        COALESCE(NULLIF(TRIM(section_1), ''), '(none)') AS primary_section,
        COUNT(*) AS stories,
        COUNT(*) FILTER (WHERE NULLIF(TRIM(body), '') IS NOT NULL) AS stories_with_body
    FROM {articles}
    GROUP BY 1
    ORDER BY stories DESC, primary_section
""").df()
by_section["primary_section"] = by_section["primary_section"].map(translate_news_category)
display(by_section)

## Forum crawl status and most-commented stories

In [ ]:
if forums:
    status = duckdb.sql(f"""
        SELECT crawl_status, COUNT(*) AS stories,
               COALESCE(SUM(observed_unique_count), 0) AS observed_comments,
               COALESCE(SUM(posting_count_discrepancy_absolute), 0) AS absolute_count_discrepancy
        FROM {forums}
        GROUP BY crawl_status
        ORDER BY stories DESC, crawl_status
    """).df()
    display(status)
else:
    display(Markdown("No forum Parquet files are available yet."))

top_stories = duckdb.sql(f"""
    SELECT a.month, a.section_1 AS primary_section, a.title,
           COALESCE(c.comment_rows, 0) AS comments, a.canonical_url
    FROM {articles} AS a
    LEFT JOIN {comment_counts} AS c USING (story_id)
    ORDER BY comments DESC, a.month, a.title
    LIMIT 15
""").df()
top_stories["primary_section"] = top_stories["primary_section"].map(translate_news_category)
display(top_stories)